In [1]:
import warnings
warnings.filterwarnings(action='ignore')

# <span style="color:red">ch1_허깅페이스</span>
- Transformers 라이브러리 내 pipeline() 함수
- Inference API(회원가입과 Access키가 있어야 함)
## 1. 텍스트 기반 감정분석(긍정/부정)

In [2]:
from transformers import pipeline
classifier = pipeline(task="text-classification",
                      model="distilbert-base-uncased-finetuned-sst-2-english")
classifier("I've been waiting for a HuggingFace course my whole life.")

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9598049521446228}]

In [3]:
classifier("이 영화는 정말 최악이야. 쓰레기같은 영화야. 너도 꼭 봤으면 좋겠어. 이 재미없는 영화를.")

[{'label': 'POSITIVE', 'score': 0.8200365304946899}]

In [4]:
result = classifier(["I've been waiting for a HuggingFace course my whole life.",
                    "I hate this so much!"])
[r.get('label') for r in result]

['POSITIVE', 'NEGATIVE']

In [5]:
classifier = pipeline(task="text-classification",
                      model="distilbert-base-uncased-finetuned-sst-2-english")
classifier(["I've been waiting for a HuggingFace course my whole life.",
            "I hate this so much!"])

Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9598049521446228},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

# 2. 제로-샷 분류(zero-shot-classification)
- 비지도 학습

In [6]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                     "facebook/bart-large-mnli")
classifier("I have a problem with my iphone that needs to be resolved asap!!",
          candidate_labels=["phone", "urgent", "tablet", "computer"])

Device set to use cpu


{'sequence': 'I have a problem with my iphone that needs to be resolved asap!!',
 'labels': ['urgent', 'phone', 'computer', 'tablet'],
 'scores': [0.5049763917922974,
  0.48007503151893616,
  0.012633666396141052,
  0.0023148944601416588]}

In [7]:
# 제시된 문장이 어떤 문장인지
classifier(
    "This is a course about the transformers library.",
    candidate_labels=["education", "business", "politics"]
)

{'sequence': 'This is a course about the transformers library.',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.9053581357002258, 0.07259627431631088, 0.02204558439552784]}

# 3. text 생성

In [8]:
generator = pipeline(task="text-generation",
                    model="gpt2") # 허깅페이스에는 gpt2까지
generator("In this course. We will teach you how to",
         pad_token_id=generator.tokenizer.eos_token_id)

Device set to use cpu


[{'generated_text': 'In this course. We will teach you how to integrate data points into your data storage in an easy way. We will provide a detailed explanation of the basic concepts of data storage and how to use it to store and use data in your applications.\n\nIf you are new to data storage, we recommend following this course. If you are not familiar with data storage, we recommend that you read the full course first.\n\nWe will also cover some of the basic concepts of data storage in this course.\n\nWe will also provide a comprehensive tutorial on writing applications using data storage.\n\nPlease note that we will be teaching you the basic concepts of data storage and the concepts of data types. This course is not a replacement for our more advanced courses.'}]

In [9]:
generator("이 과정은 다음과 같은 방법을 알려드려요. ",
         pad_token_id=generator.tokenizer.eos_token_id)

[{'generated_text': '이 과정은 다음과 같은 방법을 알려드려요. 화이말이 이오이 다인 사랑한까 있어어요. 않이을 나자 국어요. 박로이 나자 굜지 저라로 라로 라로 라로 라로 라로라로 라로라로 오로 라로라로 라로라로 라로라로. 위재어요 재어래 문라로라로 라로라로 라로라로 라�'}]

# 4. 마스크 채우기

In [10]:
unmasker = pipeline("fill-mask", "distilroberta-base")
unmasker("I'm going to hospital and meet a <mask>")

Some weights of the model checkpoint at distilroberta-base were not used when initializing RobertaForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


[{'score': 0.19275875389575958,
  'token': 3299,
  'token_str': ' doctor',
  'sequence': "I'm going to hospital and meet a doctor"},
 {'score': 0.06794668734073639,
  'token': 27321,
  'token_str': ' psychiatrist',
  'sequence': "I'm going to hospital and meet a psychiatrist"},
 {'score': 0.06435622274875641,
  'token': 16308,
  'token_str': ' surgeon',
  'sequence': "I'm going to hospital and meet a surgeon"},
 {'score': 0.05912911519408226,
  'token': 9008,
  'token_str': ' nurse',
  'sequence': "I'm going to hospital and meet a nurse"},
 {'score': 0.05705659091472626,
  'token': 1441,
  'token_str': ' friend',
  'sequence': "I'm going to hospital and meet a friend"}]

In [11]:
unmasker("Hello, I'm a <mask> girl",
        top_k=2) #top_k를 안 주면 5개

[{'score': 0.04969281330704689,
  'token': 11962,
  'token_str': ' cute',
  'sequence': "Hello, I'm a cute girl"},
 {'score': 0.035202886909246445,
  'token': 410,
  'token_str': ' little',
  'sequence': "Hello, I'm a little girl"}]

In [ ]:
# google-bert/bert-base-uncased사용을 위해 key 발부
from transformers import pipeline
unmasker = pipeline(task="fill-mask",
                   model="google-bert/bert-base-uncased")
unmasker("Hello, I'm a [MASK] model", top_k=2)

Some weights of the model checkpoint at google-bert/bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [14]:
import os
from dotenv import load_dotenv
load_dotenv()
# print(os.environ['HF_TOKEN'])

True

In [20]:
from huggingface_hub import InferenceClient
client = InferenceClient(provider="hf-inference",
                         api_key=os.environ['HF_TOKEN'])
result = client.fill_mask(
    "Hello, I'm a [MASK] model",
    model="google-bert/bert-base-uncased",
    top_k=2
)

In [22]:
[r.sequence for r in result]

["hello, i ' m a fashion model", "hello, i ' m a new model"]

In [16]:
# 다국어지원 모델도 한글 지원 만족스럽지 않을 수 있음
unmasker = pipeline("fill-mask",
                   model="bert-base-multilingual-cased") 

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Device set to use cpu
